# Wingers Features V2

Adds: progressive metrics, dribble success rate, optional age cohorts, and last-5 form trends.

In [19]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

EVENTS_PATH = Path("../data/processed/winger_events.parquet")
OUT_PATH = Path("../data/processed/winger_features_v2.parquet")
MATCHES_ROOT = Path("../data/raw/statsbomb/matches")
PLAYER_BIO_PATH = Path("../data/processed/player_bio.parquet")  # optional

df = pd.read_parquet(EVENTS_PATH)
print("winger_events shape:", df.shape)
print(df.columns.tolist())


winger_events shape: (2656669, 15)
['match_id', 'minute', 'type.name', 'player_id', 'player_name', 'team_name', 'location', 'pass.length', 'shot.statsbomb_xg', 'dribble.outcome.name', 'player_name_winger', 'team_name_winger', 'position', 'start_minute', 'end_minute']


In [20]:
for c in ["match_id", "player_id", "minute", "start_minute", "end_minute"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

def get_x(loc):
    if isinstance(loc, (list, tuple)) and len(loc) >= 1:
        try:
            return float(loc[0])
        except Exception:
            return np.nan
    return np.nan

df["x_start"] = df["location"].map(get_x) if "location" in df.columns else np.nan
df.head()


,match_id,minute,type.name,player_id,player_name,team_name,location,pass.length,shot.statsbomb_xg,dribble.outcome.name,player_name_winger,team_name_winger,position,start_minute,end_minute,x_start
0,15946,0,Pass,6581,Jonathan Rodríguez Menéndez,Deportivo Alavés,"[61.0, 40.1]",29.76995,NaN,None,Jonathan Rodríguez Menéndez,Deportivo Alavés,Left Midfield,0.0,130.0,NaN
1,15946,0,Ball Receipt*,5477,Ousmane Dembélé,Barcelona,"[36.2, 5.3]",NaN,NaN,None,Ousmane Dembélé,Barcelona,Left Midfield,0.0,130.0,NaN
2,15946,0,Ball Receipt*,5477,Ousmane Dembélé,Barcelona,"[36.2, 5.3]",NaN,NaN,None,Ousmane Dembélé,Barcelona,Left Wing,0.0,130.0,NaN
3,15946,0,Carry,5477,Ousmane Dembélé,Barcelona,"[36.2, 5.3]",NaN,NaN,None,Ousmane Dembélé,Barcelona,Left Midfield,0.0,130.0,NaN
4,15946,0,Carry,5477,Ousmane Dembélé,Barcelona,"[36.2, 5.3]",NaN,NaN,None,Ousmane Dembélé,Barcelona,Left Wing,0.0,130.0,NaN


In [21]:
minutes = (
    df[["match_id", "player_id", "start_minute", "end_minute"]]
    .drop_duplicates()
    .assign(
        start_minute=lambda x: pd.to_numeric(x["start_minute"], errors="coerce").fillna(0),
        end_minute=lambda x: pd.to_numeric(x["end_minute"], errors="coerce").fillna(130),
    )
    .assign(minutes=lambda x: (x["end_minute"] - x["start_minute"]).clip(lower=0))
    .groupby("player_id", as_index=False)["minutes"]
    .sum()
)

dribbles_attempted = (df[df["type.name"] == "Dribble"].groupby("player_id").size().rename("dribbles_attempted"))
dribbles_successful = (
    df[(df["type.name"] == "Dribble") & (df.get("dribble.outcome.name", pd.Series(index=df.index)).fillna("") == "Complete")]
    .groupby("player_id").size().rename("dribbles_successful")
)
carries = (df[df["type.name"] == "Carry"].groupby("player_id").size().rename("carries"))
passes = (df[df["type.name"] == "Pass"].groupby("player_id").size().rename("passes"))
shots = (df[df["type.name"] == "Shot"].groupby("player_id").size().rename("shots"))
xg = (df[df["type.name"] == "Shot"].groupby("player_id")["shot.statsbomb_xg"].sum().rename("xg"))

features = pd.concat([dribbles_attempted, dribbles_successful, carries, passes, shots, xg], axis=1).fillna(0).reset_index()
features = features.merge(minutes, on="player_id", how="left")
features["minutes"] = features["minutes"].fillna(0)
features.head()


,player_id,dribbles_attempted,dribbles_successful,carries,passes,shots,xg,minutes
0,2937,3.0,0.0,13.0,10.0,0.0,0.000000,130.0
1,2941,30.0,15.0,236.0,177.0,21.0,3.395412,1040.0
2,2944,8.0,5.0,87.0,130.0,4.0,0.351674,1170.0
3,2948,12.0,8.0,101.0,76.0,7.0,1.190467,780.0
4,2950,15.0,8.0,350.0,388.0,10.0,1.642851,910.0


In [22]:
has_pass_end = "pass.end_location" in df.columns
has_carry_end = "carry.end_location" in df.columns

if has_pass_end:
    def get_end_x(loc):
        if isinstance(loc, (list, tuple)) and len(loc) >= 1:
            try:
                return float(loc[0])
            except Exception:
                return np.nan
        return np.nan

    pass_df = df[df["type.name"] == "Pass"].copy()
    pass_df["x_end"] = pass_df["pass.end_location"].map(get_end_x)
    pass_df["dx"] = pass_df["x_end"] - pass_df["x_start"]
    prog_passes = pass_df[(pass_df["dx"] >= 10)].groupby("player_id").size().rename("progressive_passes")
else:
    prog_passes = (
        df[(df["type.name"] == "Pass") & (pd.to_numeric(df.get("pass.length", 0), errors="coerce").fillna(0) >= 25)]
        .groupby("player_id").size().rename("progressive_passes")
    )

if has_carry_end:
    def get_end_x2(loc):
        if isinstance(loc, (list, tuple)) and len(loc) >= 1:
            try:
                return float(loc[0])
            except Exception:
                return np.nan
        return np.nan

    carry_df = df[df["type.name"] == "Carry"].copy()
    carry_df["x_end"] = carry_df["carry.end_location"].map(get_end_x2)
    carry_df["dx"] = carry_df["x_end"] - carry_df["x_start"]
    prog_carries = carry_df[(carry_df["dx"] >= 10)].groupby("player_id").size().rename("progressive_carries")
else:
    prog_carries = (
        df[(df["type.name"] == "Carry") & (df["x_start"] >= 50) & (df["x_start"] <= 85)]
        .groupby("player_id").size().rename("progressive_carries")
    )

features = features.merge(prog_passes.reset_index(), on="player_id", how="left")
features = features.merge(prog_carries.reset_index(), on="player_id", how="left")
features[["progressive_passes", "progressive_carries"]] = features[["progressive_passes", "progressive_carries"]].fillna(0)

print("progressive_passes source:", "true_end_location" if has_pass_end else "long_pass_proxy")
print("progressive_carries source:", "true_end_location" if has_carry_end else "advanced_zone_proxy")


progressive_passes source: long_pass_proxy
progressive_carries source: advanced_zone_proxy


In [23]:
features["dribble_success_rate"] = np.where(
    features["dribbles_attempted"] > 0,
    features["dribbles_successful"] / features["dribbles_attempted"],
    0.0,
)

for col in ["dribbles_attempted","dribbles_successful","carries","progressive_carries","passes","progressive_passes","shots","xg"]:
    features[f"{col}_per90"] = np.where(features["minutes"] > 0, features[col] / features["minutes"] * 90, 0.0)

features.head()


,player_id,dribbles_attempted,dribbles_successful,carries,passes,shots,xg,minutes,progressive_passes,progressive_carries,dribble_success_rate,dribbles_attempted_per90,dribbles_successful_per90,carries_per90,progressive_carries_per90,passes_per90,progressive_passes_per90,shots_per90,xg_per90
0,2937,3.0,0.0,13.0,10.0,0.0,0.000000,130.0,1.0,0.0,0.000000,2.076923,0.000000,9.000000,0.0,6.923077,0.692308,0.000000,0.000000
1,2941,30.0,15.0,236.0,177.0,21.0,3.395412,1040.0,23.0,0.0,0.500000,2.596154,1.298077,20.423077,0.0,15.317308,1.990385,1.817308,0.293834
2,2944,8.0,5.0,87.0,130.0,4.0,0.351674,1170.0,15.0,0.0,0.625000,0.615385,0.384615,6.692308,0.0,10.000000,1.153846,0.307692,0.027052
3,2948,12.0,8.0,101.0,76.0,7.0,1.190467,780.0,11.0,0.0,0.666667,1.384615,0.923077,11.653846,0.0,8.769231,1.269231,0.807692,0.137362
4,2950,15.0,8.0,350.0,388.0,10.0,1.642851,910.0,67.0,0.0,0.533333,1.483516,0.791209,34.615385,0.0,38.373626,6.626374,0.989011,0.162480


In [24]:
match_rows = []
for f in MATCHES_ROOT.rglob("*.json"):
    try:
        data = json.loads(f.read_text(encoding="utf-8"))
    except Exception:
        continue
    if isinstance(data, list):
        for m in data:
            mid = m.get("match_id")
            md = m.get("match_date")
            if mid is not None and md is not None:
                match_rows.append({"match_id": int(mid), "match_date": md})

match_dates = pd.DataFrame(match_rows).drop_duplicates(subset=["match_id"])
match_dates["match_date"] = pd.to_datetime(match_dates["match_date"], errors="coerce")
print("match_dates rows:", len(match_dates))
match_dates.head()


match_dates rows: 3464


,match_id,match_date
0,9880,2018-04-14
1,9912,2018-04-29
2,9924,2018-05-06
3,9855,2018-03-18
4,9827,2018-03-01


In [25]:
event_by_match = (
    df.groupby(["player_id", "match_id", "type.name"]).size().unstack(fill_value=0).reset_index()
)
for c in ["Dribble", "Carry", "Pass", "Shot"]:
    if c not in event_by_match.columns:
        event_by_match[c] = 0

xg_by_match = (
    df[df["type.name"] == "Shot"]
    .groupby(["player_id", "match_id"])["shot.statsbomb_xg"]
    .sum()
    .rename("xg")
    .reset_index()
)
mins_by_match = (
    df[["player_id", "match_id", "start_minute", "end_minute"]]
    .drop_duplicates()
    .assign(start_minute=lambda x: pd.to_numeric(x["start_minute"], errors="coerce").fillna(0), end_minute=lambda x: pd.to_numeric(x["end_minute"], errors="coerce").fillna(130))
    .assign(minutes=lambda x: (x["end_minute"] - x["start_minute"]).clip(lower=0))
    .groupby(["player_id", "match_id"], as_index=False)["minutes"].sum()
)

pm = event_by_match.merge(xg_by_match, on=["player_id", "match_id"], how="left").merge(mins_by_match, on=["player_id", "match_id"], how="left")
pm["xg"] = pm["xg"].fillna(0)
pm = pm.merge(match_dates, on="match_id", how="left")

for raw_col, new_col in [("Dribble","dribbles_per90"),("Carry","carries_per90"),("Pass","passes_per90"),("Shot","shots_per90"),("xg","xg_per90")]:
    pm[new_col] = np.where(pm["minutes"] > 0, pm[raw_col] / pm["minutes"] * 90, 0.0)

pm = pm.sort_values(["player_id", "match_date", "match_id"])
season_avg = pm.groupby("player_id")[["dribbles_per90","carries_per90","passes_per90","shots_per90","xg_per90"]].mean().reset_index()
last5 = pm.groupby("player_id").tail(5).groupby("player_id")[["dribbles_per90","carries_per90","passes_per90","shots_per90","xg_per90"]].mean().reset_index()
season_avg = season_avg.rename(columns={c: f"{c}_season" for c in season_avg.columns if c != "player_id"})
last5 = last5.rename(columns={c: f"{c}_last5" for c in last5.columns if c != "player_id"})
form = season_avg.merge(last5, on="player_id", how="outer")
for base in ["dribbles_per90","carries_per90","passes_per90","shots_per90","xg_per90"]:
    form[f"{base}_trend_last5_minus_season"] = form[f"{base}_last5"].fillna(0) - form[f"{base}_season"].fillna(0)

features = features.merge(form, on="player_id", how="left")
features.head()


,player_id,dribbles_attempted,dribbles_successful,carries,passes,shots,xg,minutes,progressive_passes,progressive_carries,...,dribbles_per90_last5,carries_per90_last5,passes_per90_last5,shots_per90_last5,xg_per90_last5,dribbles_per90_trend_last5_minus_season,carries_per90_trend_last5_minus_season,passes_per90_trend_last5_minus_season,shots_per90_trend_last5_minus_season,xg_per90_trend_last5_minus_season
0,2937,3.0,0.0,13.0,10.0,0.0,0.000000,130.0,1.0,0.0,...,2.076923,9.000000,6.923077,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
1,2941,30.0,15.0,236.0,177.0,21.0,3.395412,1040.0,23.0,0.0,...,1.661538,18.553846,13.846154,1.523077,0.301810,-9.346154e-01,-1.869231,-1.471154,-0.294231,0.007976
2,2944,8.0,5.0,87.0,130.0,4.0,0.351674,1170.0,15.0,0.0,...,0.692308,8.861538,11.769231,0.276923,0.026224,7.692308e-02,2.169231,1.769231,-0.030769,-0.000828
3,2948,12.0,8.0,101.0,76.0,7.0,1.190467,780.0,11.0,0.0,...,1.384615,12.738462,9.415385,0.830769,0.127927,-2.220446e-16,1.084615,0.646154,0.023077,-0.009435
4,2950,15.0,8.0,350.0,388.0,10.0,1.642851,910.0,67.0,0.0,...,1.800000,35.307692,37.661538,0.692308,0.038781,3.164835e-01,0.692308,-0.712088,-0.296703,-0.123699


In [26]:
if PLAYER_BIO_PATH.exists():
    bio = pd.read_parquet(PLAYER_BIO_PATH)
    if "age" not in bio.columns and "dob" in bio.columns:
        bio["dob"] = pd.to_datetime(bio["dob"], errors="coerce")
        ref_date = pd.Timestamp("today").normalize()
        bio["age"] = (ref_date - bio["dob"]).dt.days / 365.25

    keep = [c for c in ["player_id", "age"] if c in bio.columns]
    if set(["player_id", "age"]).issubset(set(keep)):
        features = features.merge(bio[keep].drop_duplicates(subset=["player_id"]), on="player_id", how="left")

features["age_cohort"] = pd.cut(
    features.get("age", pd.Series(index=features.index, dtype=float)),
    bins=[0, 21, 24, 27, 30, 50],
    labels=["U21", "22-24", "25-27", "28-30", "31+"],
    include_lowest=True,
)
features["age_cohort"] = features["age_cohort"].astype("object").fillna("Unknown")
features.head()


,player_id,dribbles_attempted,dribbles_successful,carries,passes,shots,xg,minutes,progressive_passes,progressive_carries,...,carries_per90_last5,passes_per90_last5,shots_per90_last5,xg_per90_last5,dribbles_per90_trend_last5_minus_season,carries_per90_trend_last5_minus_season,passes_per90_trend_last5_minus_season,shots_per90_trend_last5_minus_season,xg_per90_trend_last5_minus_season,age_cohort
0,2937,3.0,0.0,13.0,10.0,0.0,0.000000,130.0,1.0,0.0,...,9.000000,6.923077,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,Unknown
1,2941,30.0,15.0,236.0,177.0,21.0,3.395412,1040.0,23.0,0.0,...,18.553846,13.846154,1.523077,0.301810,-9.346154e-01,-1.869231,-1.471154,-0.294231,0.007976,Unknown
2,2944,8.0,5.0,87.0,130.0,4.0,0.351674,1170.0,15.0,0.0,...,8.861538,11.769231,0.276923,0.026224,7.692308e-02,2.169231,1.769231,-0.030769,-0.000828,Unknown
3,2948,12.0,8.0,101.0,76.0,7.0,1.190467,780.0,11.0,0.0,...,12.738462,9.415385,0.830769,0.127927,-2.220446e-16,1.084615,0.646154,0.023077,-0.009435,Unknown
4,2950,15.0,8.0,350.0,388.0,10.0,1.642851,910.0,67.0,0.0,...,35.307692,37.661538,0.692308,0.038781,3.164835e-01,0.692308,-0.712088,-0.296703,-0.123699,Unknown


In [27]:
# Normalize player name from any available source column
name_candidates = [c for c in ["player_name", "player_name_x", "player_name_y"] if c in df.columns]
if name_candidates:
    names = (
        df[["player_id"] + name_candidates]
        .copy()
        .assign(player_name=lambda x: x[name_candidates].bfill(axis=1).iloc[:, 0])
        [["player_id", "player_name"]]
        .dropna(subset=["player_name"])
        .drop_duplicates(subset=["player_id"])
    )
    features = features.merge(names, on="player_id", how="left")

# Ensure a clean player_name column always exists
if "player_name" not in features.columns:
    features["player_name"] = features["player_id"].astype(str)

features = features[features["minutes"] >= 300].copy()
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
features.to_parquet(OUT_PATH, index=False)

print("Saved:", OUT_PATH.resolve())
print("Shape:", features.shape)
features[["player_id", "player_name", "minutes"]].head()


Saved: C:\Users\ismail\football-scouting-platform\data\processed\winger_features_v2.parquet
Shape: (1438, 36)


,player_id,player_name,minutes
1,2941,Ismaïla Sarr,1040.0
2,2944,Benjamin Corgnet,1170.0
3,2948,Nabil Fekir,780.0
4,2950,Jonas Martin,910.0
6,2955,Ronny Rodelin,4030.0
